<a href="https://colab.research.google.com/github/DariaDubovska/quora-duplicate-question-detection/blob/main/Quora_duplicates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Libraries importing

In [2]:
import pandas as pd
import numpy as np
import re
import html
import spacy
from collections import Counter
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, classification_report
from transformers import BertTokenizer, BertModel
import torch
from sklearn.metrics.pairwise import cosine_similarity

import os
import pickle

In [3]:
RANDOMSEED = 42

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/ML Projects/Quora duplicates"
cache_file = os.path.join(PROJECT_DIR, "embeddings/bert_embeddings.pkl")
chunks_dir = os.path.join(PROJECT_DIR, "embeddings/bert_cache_chunks")
feature_file = os.path.join(PROJECT_DIR, "data/processed/quora_with_bert_cosine_similarity.csv")

## Data Loading

In [4]:
df = pd.read_csv("quora_question_pairs_train.csv.zip", index_col=0)
pd.set_option("display.max_colwidth", None)
df.head()

,qid1,qid2,question1,question2,is_duplicate
id,,,,,
332278,459256,459257,The Iliad and the Odyssey in the Greek culture?,How do I prove that the pairs of three independent variables is also independent?,0
196656,297402,297403,What is practical management and what is strategic management?,What are the practical aspects of strategic management?,0
113125,184949,184950,How useful is MakeUseOf Answers?,"Is there any Q&A site that is not Yahoo answers, where hate speech is allowed?",0
266232,101283,163744,Which is the best place to reside in India and Why?,Which ia the best place to visit in India?,0
122738,17811,27517,Why do so many people ask questions on Quora that can be easily answered by any number of legitimate sources on the Web? Have they not heard of Google or Bing?,Why don't many people posting questions on Quora check Google first?,1


# Explonatory Data Analysis

In [5]:
df.shape

(323432, 5)

In [ ]:
df.isnull().sum()

,0
qid1,0
qid2,0
question1,1
question2,2
is_duplicate,0


In [ ]:
df[df["question1"].isna()]

,qid1,qid2,question1,question2,is_duplicate
id,,,,,
363362,493340,493341,NaN,My Chinese name is Haichao Yu. What English name is most suitable for me considering the pronounciation of my Chinese name?,0


In [ ]:
df[df["question2"].isna()]

,qid1,qid2,question1,question2,is_duplicate
id,,,,,
201841,303951,174364,How can I create an Android app?,NaN,0
105780,174363,174364,How can I develop android app?,NaN,0


**Dropping entries with missing values**

In [8]:
df.dropna(subset=["question1", "question2"], inplace=True)
df.reset_index(drop=True, inplace=True)

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
((df["question1"].str.strip() == "") |
 (df["question2"].str.strip() == "")).sum()

np.int64(0)

**There is no duplicates or empty strings in dataset**

In [ ]:
df["is_duplicate"].value_counts(normalize=True)

,proportion
is_duplicate,
0,0.6308
1,0.3692


In [ ]:
print("Percentage Distribution of",(df["is_duplicate"].value_counts()/df["is_duplicate"].count())*100)

Percentage Distribution of is_duplicate
0    63.079996
1    36.920004
Name: count, dtype: float64


In [9]:
questions = pd.concat([df["question1"], df["question2"]], ignore_index=True)
questions.str.len().describe()

,0
count,646858.000000
mean,59.827502
std,31.968420
min,1.000000
25%,39.000000
50%,51.000000
75%,72.000000
max,1169.000000


In [ ]:
questions.str.len().quantile([0.95, 0.99, 0.999])

,0
0.950,125.00
0.990,156.00
0.999,270.14


**Most questions are short: the mean length is around 60 characters, and 99% of questions are shorter than roughly 270 characters. This supports using `max_length=128` for BERT tokenization as a practical trade-off.**

In [ ]:
print("Number of unique questions:", questions.nunique())

Number of unique questions: 449361


In [ ]:
print("Number of all questions: ", questions.shape[0])

Number of all questions:  646858


## Data Split

In [ ]:
X = df.drop(columns=["is_duplicate"])
y = df['is_duplicate']

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, stratify = y,
                                                  random_state=RANDOMSEED)

# BERT EMBEDDING

This section uses pretrained `bert-base-uncased` embeddings as a feature extractor. The model is not fine-tuned here. Cosine similarity between question embeddings is used as a simple semantic baseline.

In [ ]:
!pip install transformers torch --quiet

In [ ]:
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model = bert_model.to(device)
bert_model.eval()

device

device(type='cuda')

## BERT embedding cache

This section loads already computed BERT embeddings from Google Drive.

In [ ]:
if os.path.exists(cache_file):
    with open(cache_file, "rb") as f:
        embedding_dict = pickle.load(f)
    print(f"Loaded {len(embedding_dict)} embeddings.")
else:
    embedding_dict = {}
    print("Creating new cache.")

Loaded 449361 embeddings.


### Compute missing embeddings only

Run this cell only if `Questions to process` is greater than 0. New embeddings are saved in chunk files, so progress is not lost if Colab disconnects.

In [ ]:
def get_sentence_embeddings_batch(texts, max_length=128):
    texts = [str(text) for text in texts]

    inputs = bert_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = bert_model(**inputs)

    last_hidden_states = outputs.last_hidden_state

    attention_mask = inputs["attention_mask"]
    mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_states.size()).float()

    embeddings = torch.sum(last_hidden_states * mask_expanded, dim=1) / torch.clamp(
        mask_expanded.sum(dim=1),
        min=1e-9
    )

    return embeddings.cpu().numpy()

In [ ]:
questions_to_process = [q for q in questions.unique() if q not in embedding_dict]
print(f"Questions to process: {len(questions_to_process)}")

In [ ]:
os.makedirs(chunks_dir, exist_ok=True)

batch_size = 64
chunk_size = 5000

chunk_id = len([f for f in os.listdir(chunks_dir) if f.endswith(".pkl")])
current_chunk = {}

for batch_start in tqdm(range(0, len(questions_to_process), batch_size)):
    batch_questions = questions_to_process[batch_start:batch_start + batch_size]

    batch_embeddings = get_sentence_embeddings_batch(batch_questions)

    for question, embedding in zip(batch_questions, batch_embeddings):
        embedding_dict[question] = embedding
        current_chunk[question] = embedding

    if len(current_chunk) >= chunk_size:
        chunk_path = os.path.join(chunks_dir, f"chunk_{chunk_id}.pkl")

        with open(chunk_path, "wb") as f:
            pickle.dump(current_chunk, f)

        print(f"Saved {chunk_path} with {len(current_chunk)} embeddings")

        current_chunk = {}
        chunk_id += 1

if len(current_chunk) > 0:
    chunk_path = os.path.join(chunks_dir, f"chunk_{chunk_id}.pkl")

    with open(chunk_path, "wb") as f:
        pickle.dump(current_chunk, f)

    print(f"Saved final {chunk_path} with {len(current_chunk)} embeddings")

In [ ]:
with open(cache_file, "wb") as f:
    pickle.dump(embedding_dict, f)

print(f"Main cache updated: {len(embedding_dict)} embeddings")

Main cache updated: 449361 embeddings


## BERT + Cosine similarity Experiment

In [ ]:
def cosine_similarity_embeddings(emb1, emb2):
     return cosine_similarity(
        emb1.reshape(1, -1),
        emb2.reshape(1, -1)
    )[0, 0]

In [ ]:
tqdm.pandas()

df["bert_cosine_similarity"] = df.progress_apply(
    lambda row: cosine_similarity_embeddings(
        embedding_dict[str(row["question1"])],
        embedding_dict[str(row["question2"])]
    ),
    axis=1
)

df[["question1", "question2", "is_duplicate", "bert_cosine_similarity"]].head()

In [ ]:
train_scores = df.loc[X_train.index, "bert_cosine_similarity"]
valid_scores = df.loc[X_valid.index, "bert_cosine_similarity"]

best_threshold = None
best_f1 = -1

for threshold in np.arange(0.1, 1.0, 0.01):
    train_preds = (train_scores >= threshold).astype(int)
    f1 = f1_score(y_train, train_preds)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

valid_preds = (valid_scores >= best_threshold).astype(int)
valid_probs = (valid_scores + 1) / 2

print(f"Best threshold from train: {best_threshold:.2f}")
print(f"Train F1 at best threshold: {best_f1:.4f}")
print("Validation accuracy:", accuracy_score(y_valid, valid_preds))
print("Validation precision:", precision_score(y_valid, valid_preds))
print("Validation recall:", recall_score(y_valid, valid_preds))
print("Validation F1:", f1_score(y_valid, valid_preds))
print("Validation ROC-AUC:", roc_auc_score(y_valid, valid_probs))
print()
print(classification_report(y_valid, valid_preds))

Best threshold from train: 0.83
Train F1 at best threshold: 0.6425
Validation accuracy: 0.6512537488792011
Validation precision: 0.5167227039462042
Validation recall: 0.855874717360355
Validation F1: 0.6443985560932549
Validation ROC-AUC: 0.7454657120871406

              precision    recall  f1-score   support

           0       0.86      0.53      0.66     40804
           1       0.52      0.86      0.64     23882

    accuracy                           0.65     64686
   macro avg       0.69      0.69      0.65     64686
weighted avg       0.74      0.65      0.65     64686



**Conclusions**
* Sentence-BERT embeddings successfully capture semantic similarity between questions.
* Using cosine similarity alone provides reasonable discrimination between duplicate and non-duplicate question pairs (ROC-AUC ≈ 0.75).
* The model achieves high recall (≈0.86), meaning that most duplicate questions are correctly identified.
* However, the relatively low precision indicates that many non-duplicate pairs are incorrectly classified as duplicates.
* These results suggest that cosine similarity is a strong semantic feature, but it is insufficient as a standalone classifier.
* In the following experiments, the cosine similarity score will be used as an additional feature together with other text-based and engineered features.

In [ ]:
df[["question1", "question2", "is_duplicate", "bert_cosine_similarity"]].to_csv(feature_file, index=False)

print("Saved cosine similarity feature to:")
print(feature_file)

Saved cosine similarity feature to:
/content/drive/MyDrive/ML Course/Homeworks/Quora duplicates/quora_with_bert_cosine_similarity.csv
